# 01 — The canonical journey: a variant to a ranked prime-editing design

**AlleleForge is a research tool. It is not a medical device and provides no medical advice.**
Off-target nominations are computational and must be experimentally validated.

This notebook walks the variant-first journey end to end on the **flagship** chemistry, prime
editing — the one where AlleleForge unifies all four axes no single open-source tool combines:
variant input, ML efficiency with honest out-of-distribution flagging, intended-vs-byproduct
outcome prediction, and a population-aware off-target report over **both** nicks.

It runs entirely on a small synthetic reference with the weight-free baselines, so it executes in
CI with no downloads. Swapping in the real PRIDICT2.0 / off-target backbones is a model-zoo change,
not a code change.

In [ ]:
import tempfile
from pathlib import Path

from alleleforge.genome.reference import ReferenceGenome
from alleleforge.variant.resolver import resolve
from alleleforge.types.edit import EditIntent
from alleleforge.types.sequence import GenomicInterval, Strand
from alleleforge.design.prime import design_prime

# A small synthetic locus with an NGG pegRNA PAM and an opposite-strand PE3b ngRNA PAM.
seq = list("AT" * 70)
seq[63:66] = list("TGG")  # plus pegRNA PAM (nick 10 nt 5' of the edit)
seq[55:58] = list("CCA")  # minus ngRNA PAM whose seed spans the edit (PE3b)
fasta = Path(tempfile.mkdtemp()) / "locus.fa"
fasta.write_text(">chr2\n" + "".join(seq) + "\n")
reference = ReferenceGenome(fasta, build="hg38")

## 1. Resolve the variant

Any input form (ClinVar accession, dbSNP rsID, HGVS, VCF, raw coordinates) normalizes to one
canonical, left-aligned, reference-validated `Variant` with a working interval.

In [ ]:
ref_base = str(reference.fetch(GenomicInterval(chrom="chr2", start=70, end=71, strand=Strand.PLUS)))
resolved = resolve(f"chr2:71:{ref_base}>C", reference=reference)
print("variant:", resolved.variant, "| working interval:", resolved.working_interval)

## 2. Design pegRNAs and rank them

`design_prime` enumerates pegRNAs (PBS 8–17, RTT 7–34 covering the edit + ≥5 nt homology, a
tevopreQ1 epegRNA motif, and a PE3/PE3b nicking guide), scores efficiency and outcome with
calibrated uncertainty, runs the off-target engine on **both** nicks, and returns ranked candidates.

In [ ]:
candidates = design_prime(resolved, EditIntent.INSTALL, reference=reference, max_candidates=5)
top = candidates[0]
peg = top.pegrna
print("chemistry        :", top.chemistry.value)
print("pegRNA spacer    :", peg.spacer.sequence, "on", peg.placement.strand.value, "strand")
print("PBS / RTT        :", len(peg.pbs), "/", len(peg.rtt), f"(+{peg.rtt_homology_3prime} homology)")
print("epegRNA motif    :", peg.three_prime_motif.value)
ng = peg.nicking_guide
print("nicking guide    :", "PE3b (seed-disrupting)" if ng and ng.seed_disrupting else "PE3")

## 3. The four axes, made honest

Every candidate carries a calibrated efficiency interval (never a bare float), an
intended-vs-byproduct distribution, and an ancestry-stratified off-target report — and flags when
the efficiency model is out of its training distribution.

In [ ]:
eff = top.efficiency
print(f"efficiency       : {eff.value:.2f}  80% interval {tuple(round(x, 2) for x in eff.interval)}")
print("  in-distribution:", eff.in_distribution, "(OOD flag is honest about HEK293T/K562 training)")
print("outcome (intended vs. byproduct):")
for a in top.outcome.alleles:
    mark = "  <- intended" if a.is_intended else ""
    print(f"  {a.allele:<24} {a.probability:.2f}{mark}")
print("off-target sites (both nicks merged):", top.offtarget.n_sites)
print("flags            :", top.flags)

Swap the synthetic locus for a real ClinVar accession against an hg38 reference, supply a gnomAD
database for population-aware off-target, and load the trained PRIDICT2.0 weights through the model
zoo — the call shape is identical. The honesty (calibrated intervals, the OOD flag, the
ancestry-stratified off-target report) is the product.